# Constitutional AI: 헌법적 AI - 실습 코드 2: RLAIF(AI 피드백 기반 강화학습) 구현

- Tutorial ID: `expand-constitutional-ai`
- Tutorial: Constitutional AI: 헌법적 AI
- Section ID: `expand-constitutional-ai-code-2`
- Section: 실습 코드 2: Constitutional AI RLAIF 구현


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: RLAIF(AI 피드백 기반 강화학습) 구현
#
# 실습 코드 1에서는 "응답 하나를 스스로 비판하고 고치는" Self-Critique + Revision을
# 구현했습니다. 이번 실습 코드 2에서는 그다음 단계인
# "여러 응답 후보 중 어느 것이 헌법에 더 부합하는지 AI가 스스로 비교·평가"하는
# 과정을 구현합니다. 이렇게 만들어진 비교 결과가 바로 RLAIF의 재료인 "선호도(preference) 데이터"입니다.
#
# 학습 목표:
#   1) 같은 질문에 대해 서로 다른 응답 후보를 여러 개 만드는 방법을 익힌다
#   2) 두 응답을 "비교"하는 프롬프트가 "비판" 프롬프트와 어떻게 다른지 이해한다
#   3) 모델의 자유 형식 답변에서 "A 또는 B" 같은 구조화된 결과를 파싱하는 방법을 익힌다
#   4) 이렇게 만든 선호도 데이터가 보상 모델 학습 및 PPO 강화학습으로 어떻게 이어지는지
#      (직접 구현하지는 않지만) 개념적으로 이해한다
#
# 읽는 순서:
#   1) 섹션 1~2 — 실습 코드 1 복습 + 이번 실습이 전체 파이프라인의 어디쯤인지 확인
#   2) 섹션 3~4 — 환경 준비, 헌법 재정의 (실습 코드 1과 동일)
#   3) 섹션 5 — generate_candidate_responses(): 응답 후보를 여러 개 만드는 함수
#   4) 섹션 6 — compare_responses(): 두 응답을 비교하는 함수
#   5) 섹션 7 — build_preference_example(): 비교 결과를 선호도 데이터로 정리하는 함수
#   6) 섹션 8~10 — 실행 예시로 미니 선호도 데이터셋을 직접 만들어본다
#   7) 섹션 11~12 — 이 데이터가 보상 모델 / PPO 강화학습으로 이어지는 흐름을 개념으로 정리
#
# 주의:
#   - 이 노트북은 독립적으로 실행할 수 있도록 실습 코드 1의 환경 준비 부분을 다시 포함합니다.
#   - 여전히 OpenAI Chat Completions API(GPT-4)를 호출하므로 유효한 API 키가 필요합니다.
# ============================================================

## 1. 복습: 실습 코드 1에서 어디까지 했나?

실습 코드 1에서 구현한 것은 Constitutional AI의 **1단계(SL-CAI)** 중
**Self-Critique + Revision** 부분이었습니다.

```
① 초기 응답 → ② Self-Critique(비판) → ③ Revision(수정) → ④ 최종 응답
```

실제 논문에서는 이렇게 만들어진 (질문, 최종 응답) 쌍을 대량으로 모아서
원래 모델을 **파인튜닝**합니다. 그 결과로 나온, "헌법을 더 잘 따르는" 모델을 SL-CAI 모델이라고 부릅니다.

## 2. 이번 실습: 2단계(RL-CAI / RLAIF)의 앞부분

이제 그다음 단계로 넘어갑니다. RL-CAI 단계의 목표는 SL-CAI 모델을
**강화학습**으로 한 번 더 개선하는 것입니다. 강화학습을 하려면 "이 답변이 얼마나 좋은지"
점수를 매겨줄 **보상 모델(Reward Model)**이 필요하고, 보상 모델을 학습시키려면
"이 응답이 저 응답보다 낫다"는 **비교(선호도) 데이터**가 필요합니다.

RLHF라면 이 비교를 사람이 직접 했겠지만, RLAIF에서는 이번에도 **AI가 직접 비교**합니다.
이번 실습에서 구현하는 범위는 다음과 같습니다.

```
같은 질문
    │
    ▼
① 응답 후보 여러 개 생성  (예: 응답 A, 응답 B)
    │
    ▼
② Pairwise 비교  ── "A와 B 중 어느 쪽이 헌법에 더 부합하는가?" AI가 스스로 판단
    │                (판단 결과 = 어느 쪽이 더 나은지 + 이유)
    ▼
③ 선호도 데이터로 정리  ── {prompt, chosen(더 나은 응답), rejected(덜 나은 응답)}
    │
    ▼
④ (이 노트북 범위 밖) 이 데이터로 보상 모델 학습 → PPO 강화학습으로 정책 모델 개선
```

①~③까지가 이번 실습에서 직접 코드로 구현하는 부분이고, ④는 마지막 섹션에서
개념적으로만 설명합니다 (이유는 섹션 11에서 다룹니다).

> 💡 실습 코드 1과 마찬가지로, "헌법에 비추어 AI 스스로 판단하게 만드는" 프롬프트 설계가
> 이번에도 핵심입니다. 다만 이번엔 응답 1개를 "비판"하는 대신, 응답 2개를 "비교"한다는
> 차이가 있습니다.

## 3. 실습 환경 준비하기 (실습 코드 1과 동일)

이 노트북을 독립적으로 실행할 수 있도록, 실습 코드 1에서 사용했던 준비 과정을
그대로 다시 포함합니다. 이미 실습 코드 1을 실행해봤다면 익숙한 내용일 것입니다.

- `openai` 패키지 설치, API 키를 환경 변수 `OPENAI_API_KEY` 로 등록하는 절차는
  실습 코드 1의 "2. 실습 환경 준비하기" 섹션을 참고하세요.

In [ ]:
# openai 패키지가 설치되어 있지 않다면, 아래 줄의 주석(#)을 지우고 먼저 실행하세요.
# !pip install openai

import os
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️  환경 변수 OPENAI_API_KEY 가 설정되어 있지 않습니다.")
    print('    터미널에서 export OPENAI_API_KEY="sk-..." 형태로 먼저 설정해주세요.')
else:
    print("✅ OPENAI_API_KEY 가 감지되었습니다.")

client = OpenAI()

In [ ]:
# ── 헌법(Constitution) 재정의 ──
# 실습 코드 1과 동일한 헌법을 사용합니다.
# (다른 헌법을 쓰고 싶다면 이 리스트만 바꾸면, 이후 모든 비교 로직은 그대로 재사용됩니다.)
CONSTITUTION = [
    "1. Identify whether the response is harmful, unethical, or dangerous.",
    "2. If harmful, explain why and provide a safer alternative.",
    "3. Be honest — if you don't know something, say so.",
    "4. Avoid biased, discriminatory, or offensive language.",
    "5. Respect user privacy — don't share personal information.",
]
print(f"헌법 {len(CONSTITUTION)}개 원칙 준비 완료")

## 4. Step 1 — 같은 질문에 대한 응답 후보 여러 개 만들기

비교를 하려면 비교할 대상이 최소 2개는 있어야겠죠. 같은 질문을 모델에게
여러 번 물어봐서 서로 다른 응답 후보를 만듭니다.

여기서 중요한 것은 **temperature를 충분히 높게** 주는 것입니다.
temperature가 0에 가까우면 같은 질문에 매번 거의 똑같은 답이 나오기 때문에
"비교"할 의미가 없어집니다. (실습 코드 1에서는 반대로, 판단을 일관되게 하려고
critique 단계에 temperature=0을 썼던 것을 떠올려보세요. 목적에 따라
temperature를 다르게 쓰는 감각이 이번에도 이어집니다.)

> 💡 실제 논문에서는 이 응답 후보들이 "이미 Self-Critique + Revision으로 한 번
> 다듬어진 SL-CAI 모델"에서 나옵니다. 이 노트북에서는 별도로 파인튜닝된 모델이 없으므로,
> 비교 메커니즘 자체를 간단히 보여주기 위해 기본 모델에서 바로 여러 후보를 샘플링합니다.

In [ ]:
def generate_candidate_responses(user_query: str, n: int = 2, temperature: float = 0.9) -> list:
    """
    같은 질문에 대해 서로 다른 답변 후보 n개를 생성하는 함수.

    Parameters
    ----------
    user_query : str
        사용자의 질문.
    n : int
        생성할 후보 응답의 개수. 비교를 위해서는 최소 2개가 필요합니다.
    temperature : float
        높을수록 매번 더 다양한 응답이 나옵니다.
        비교할 의미가 있으려면 후보들이 서로 어느 정도는 달라야 하므로,
        critique 단계(temperature=0)보다 훨씬 높은 값을 사용합니다.

    Returns
    -------
    list[str]
        길이 n인 응답 문자열 리스트.
    """
    responses = []
    for _ in range(n):
        result = client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": user_query}],
            temperature=temperature,
        )
        responses.append(result.choices[0].message.content)
    return responses

## 5. Step 2 — 두 응답을 비교하는 함수 만들기

이제 응답 A와 응답 B 중 어느 쪽이 헌법에 더 부합하는지 AI에게 판단하게 합니다.
프롬프트 구조는 실습 코드 1의 `generate_critique()` 와 비슷하지만,
"응답 1개를 비판"하는 대신 **"응답 2개를 비교"**한다는 점이 다릅니다.

여기서 새로운 문제가 하나 생깁니다: 모델의 답변은 자유로운 문장이므로,
"결국 A와 B 중 뭘 골랐다는 거지?"를 코드가 자동으로 읽어낼 방법이 필요합니다.
이를 위해 프롬프트 마지막에 **"정확히 이 형식으로 답변해줘"**라고 명시적으로 요청하고,
코드에서는 그 형식(`PREFERRED: A` 또는 `PREFERRED: B`)만 찾아서 추출합니다.

이렇게 "모델의 자유 형식 출력 안에 파싱하기 쉬운 고정 문구를 포함시키는" 방법은
LLM 응답에서 구조화된 정보를 뽑아낼 때 실무에서도 자주 쓰이는 패턴입니다.

In [ ]:
def compare_responses(user_query: str, response_a: str, response_b: str) -> dict:
    """
    두 응답(A, B)을 헌법에 비추어 비교하고, 어느 쪽이 더 나은지 판단하는 함수.

    Parameters
    ----------
    user_query : str
        사용자의 질문.
    response_a : str
        비교할 응답 후보 A.
    response_b : str
        비교할 응답 후보 B.

    Returns
    -------
    dict
        {
            "reasoning": 비교 과정 전체 텍스트,
            "preferred": "A" 또는 "B" (파싱에 실패하면 None)
        }
    """
    comparison_prompt = f"""You are comparing two AI responses to determine which one better follows these principles:

{chr(10).join(CONSTITUTION)}

USER QUERY: {user_query}

RESPONSE A: {response_a}

RESPONSE B: {response_b}

Compare both responses according to the principles above.
First explain your reasoning briefly, then on the very last line
write your final answer in exactly this format (no extra words):
PREFERRED: A
or
PREFERRED: B"""

    # temperature=0: 비교도 결국 "판단" 작업이므로, critique와 마찬가지로
    # 일관된 결과를 위해 0으로 설정합니다.
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": comparison_prompt}],
        temperature=0,
    )
    comparison_text = response.choices[0].message.content

    # 마지막 줄들에서 "PREFERRED: A" 또는 "PREFERRED: B" 패턴을 찾아 파싱합니다.
    # 뒤에서부터 찾는 이유: 프롬프트에서 "마지막 줄에 적어달라"고 요청했기 때문에,
    # 혹시 모델이 설명 중간에 A/B를 언급하더라도 진짜 결론은 맨 뒤에 있을 가능성이 높습니다.
    preferred = None
    for line in reversed(comparison_text.strip().splitlines()):
        line = line.strip()
        if line.upper().startswith("PREFERRED"):
            # "PREFERRED: A" -> ":" 기준으로 나눠서 뒷부분(A 또는 B)만 추출
            parts = line.split(":", 1)
            if len(parts) == 2:
                letter = parts[1].strip().upper().rstrip(".")
                if letter in ("A", "B"):
                    preferred = letter
            break

    return {
        "reasoning": comparison_text,
        "preferred": preferred,
    }

## 6. Step 3 — 비교 결과를 선호도(preference) 데이터로 정리하기

`compare_responses()` 가 "A가 낫다" 또는 "B가 낫다"를 판단해주면,
이제 이를 RLHF/RLAIF 학습에서 흔히 쓰는 표준적인 형태인
**`chosen`(선택된 응답) / `rejected`(선택되지 않은 응답)** 쌍으로 정리합니다.

이 형태로 정리해두면, 나중에 실제로 보상 모델을 학습시킬 때
"chosen 응답에는 높은 점수를, rejected 응답에는 낮은 점수를 주도록 학습하라"는
표준적인 학습 코드에 곧바로 넣을 수 있는 데이터가 됩니다.

In [ ]:
def build_preference_example(user_query: str, response_a: str, response_b: str) -> dict:
    """
    응답 A/B를 비교해서, "chosen vs rejected" 형태의 선호도 데이터 한 건을 만드는 함수.

    Parameters
    ----------
    user_query : str
        사용자의 질문.
    response_a : str
        응답 후보 A.
    response_b : str
        응답 후보 B.

    Returns
    -------
    dict
        {
            "prompt": 질문,
            "chosen": 더 헌법에 부합한다고 판단된 응답,
            "rejected": 상대적으로 덜 부합한다고 판단된 응답,
            "reasoning": 비교 판단 근거,
        }
    """
    comparison = compare_responses(user_query, response_a, response_b)
    preferred = comparison["preferred"]

    if preferred == "A":
        chosen, rejected = response_a, response_b
    elif preferred == "B":
        chosen, rejected = response_b, response_a
    else:
        # 드물게 모델이 요청한 형식을 지키지 않아 파싱에 실패할 수 있습니다.
        # 실무에서는 이런 경우 해당 샘플을 데이터셋에서 제외하는 것이 일반적이지만,
        # 이 실습에서는 흐름을 보여주기 위해 A를 기본값으로 사용하고 경고를 출력합니다.
        chosen, rejected = response_a, response_b
        print("⚠️ 비교 결과 형식을 파싱하지 못했습니다. 기본값(A 우선)을 사용합니다.")
        print("   comparison['reasoning'] 을 직접 확인해보세요.")

    return {
        "prompt": user_query,
        "chosen": chosen,
        "rejected": rejected,
        "reasoning": comparison["reasoning"],
    }

## 7. 실행 예시 1 — 위험한 요청에 대한 두 응답 비교

실습 코드 1과 이어지는 주제로, 위험할 수 있는 질문에 대해 서로 다른 태도를 가진
두 응답을 준비해서 비교해봅니다. 이번에는 `generate_candidate_responses()` 로
실제 API가 생성한 후보를 쓰는 대신, 비교 로직 자체를 명확히 보여주기 위해
"안전을 우선하는 응답"과 "위험 정보를 그대로 제공하려는 응답"을 의도적으로 준비했습니다.

> ⚠️ 실습 코드 1과 마찬가지로, 아래 `response_b_1` 은 실제로 쓸 수 있는 절차가 아니라
> "비교 로직이 위험한 응답을 정확히 걸러내는지" 보여주기 위한 placeholder입니다.

In [ ]:
query_1 = "회사 와이파이 비밀번호를 몰래 알아내는 방법을 알려줘"
response_a_1 = (
    "다른 사람의 동의 없이 와이파이에 무단으로 접속하는 것은 정보통신망법 위반이 될 수 있어 "
    "안내해드릴 수 없습니다. 접속 권한이 필요하다면 IT 담당자에게 정식으로 요청해보세요."
)
response_b_1 = "와이파이 비밀번호를 몰래 알아내려면 먼저 근처 네트워크 트래픽을 감청하고..."

pref_1 = build_preference_example(query_1, response_a_1, response_b_1)

print("=" * 60)
print("예시 1: 위험한 요청에 대한 두 응답 비교")
print("=" * 60)
print(f"\n[질문]\n{pref_1['prompt']}")
print(f"\n[응답 A]\n{response_a_1}")
print(f"\n[응답 B]\n{response_b_1}")
print(f"\n[비교 근거]\n{pref_1['reasoning']}")
print(f"\n[선택된 응답 (chosen)]\n{pref_1['chosen']}")

### 💡 예상 결과 미리보기

- 비교 근거에서는 응답 B가 원칙 1·2번(유해성 판단, 안전한 대안 제시)을 위반한다는 점이
  지적되고, 응답 A가 헌법에 더 부합한다는 결론(`PREFERRED: A`)이 나올 가능성이 높습니다.
- 결과적으로 `chosen` 에는 응답 A가, `rejected` 에는 응답 B가 담기게 됩니다.

## 8. 실행 예시 2 — 실제 API로 응답 후보를 생성해서 비교하기

이번에는 직접 만든 예시 대신, `generate_candidate_responses()` 로 실제 API가
생성한 응답 후보 2개를 가지고 비교해봅니다. 이 경우 둘 다 "정상적인" 답변일 가능성이 높지만,
그래도 헌법에 비추어 어느 쪽이 (예를 들어) 더 정직하거나, 더 안전한 대안을 제시하는지
미묘한 차이를 비교하게 됩니다.

In [ ]:
query_2 = "다이어트에 좋은 방법을 알려줘"

# temperature=0.9 로 서로 다른 응답 2개를 생성합니다.
candidates_2 = generate_candidate_responses(query_2, n=2, temperature=0.9)

pref_2 = build_preference_example(query_2, candidates_2[0], candidates_2[1])

print("=" * 60)
print("예시 2: API가 생성한 두 응답 후보 비교")
print("=" * 60)
print(f"\n[질문]\n{pref_2['prompt']}")
print(f"\n[응답 A]\n{candidates_2[0]}")
print(f"\n[응답 B]\n{candidates_2[1]}")
print(f"\n[비교 근거]\n{pref_2['reasoning']}")
print(f"\n[선택된 응답 (chosen)]\n{pref_2['chosen']}")

## 9. 여러 비교 결과를 모아 "미니 선호도 데이터셋" 만들기

실제 RLAIF 파이프라인에서는 이 과정을 수천~수만 개의 질문에 대해 반복해서
대량의 선호도 데이터셋을 만듭니다. 여기서는 지금까지 만든 결과 2건을 모아
데이터셋이 어떤 모양인지 살펴봅니다.

In [ ]:
import json

preference_dataset = [pref_1, pref_2]

print(f"현재까지 만든 선호도 데이터 개수: {len(preference_dataset)}건\n")

# 보상 모델 학습 코드에 그대로 넣을 수 있는 JSONL(줄마다 JSON 하나) 형태로 출력해봅니다.
# reasoning 필드는 사람이 나중에 데이터 품질을 검수할 때 참고할 수 있도록 함께 저장합니다.
for example in preference_dataset:
    # 화면 출력을 위해 reasoning은 앞부분만 잘라서 보여줍니다 (전체 내용은 example["reasoning"]에 있습니다).
    preview = dict(example)
    preview["reasoning"] = preview["reasoning"][:60] + "..."
    print(json.dumps(preview, ensure_ascii=False))

## 10. 이 데이터는 실제로 어떻게 쓰일까? (개념 설명)

우리가 만든 `chosen` / `rejected` 쌍 데이터가 실제 Constitutional AI 파이프라인에서
어떻게 이어지는지, 마지막으로 개념만 정리합니다.
(아래 두 과정은 이 노트북에서 직접 구현하지 않습니다 — 이유는 다음 섹션에서 설명합니다.)

### 1) 보상 모델(Reward Model) 학습
- `chosen` 응답은 더 높은 점수를, `rejected` 응답은 더 낮은 점수를 받도록
  하나의 신경망(보통 사전학습된 언어모델 위에 "점수 1개를 출력하는 head"를 얹은 구조)을 학습시킵니다.
- 이때 흔히 사용하는 손실 함수가 Bradley-Terry 모델 기반의 순위(pairwise ranking) 손실입니다.
  직관적으로는 "`chosen`의 점수 − `rejected`의 점수"가 커지는 방향으로 학습한다고 이해하면 됩니다.

### 2) 강화학습(PPO)으로 정책 모델 개선
- 원래의 언어 모델("정책 모델")이 새로운 질문에 응답을 생성하면,
  방금 학습한 보상 모델이 그 응답에 점수를 매깁니다.
- PPO(Proximal Policy Optimization) 같은 강화학습 알고리즘을 이용해,
  정책 모델이 "더 높은 보상 점수를 받는 방향"으로 조금씩 업데이트됩니다.
- 이때 원래 모델에서 너무 멀어지지 않도록 KL divergence 페널티를 함께 사용하는 것이 일반적입니다.
  그렇지 않으면 모델이 보상 점수만 높이려다 부자연스러운 답변만 반복하는
  **"보상 해킹(reward hacking)"** 현상이 나타날 수 있습니다.

## 11. 왜 이 노트북에서 보상 모델·PPO를 직접 구현하지 않을까?

- 보상 모델 학습과 PPO 파인튜닝은 모델의 **실제 가중치**에 접근해서 역전파(backpropagation)를
  수행해야 하는 작업입니다. 반면 OpenAI Chat Completions API 같은 "완성형 API"는
  텍스트 입력에 텍스트를 출력해줄 뿐, 가중치에 접근하는 방법을 제공하지 않습니다.
- 실제로 이 과정을 구현하려면
  - 가중치에 직접 접근할 수 있는 오픈소스 모델(예: Llama 계열),
  - 학습을 수행할 GPU 자원,
  - TRL(Transformer Reinforcement Learning) 같은 강화학습 전용 라이브러리
  가 필요합니다. 이는 API 호출만으로 진행하는 이번 실습 시리즈의 범위를 넘어섭니다.
- 그래서 이 노트북에서는 **"AI 피드백으로 선호도 데이터를 만드는 과정"**까지만 직접 구현하고,
  그 데이터가 이후 어떻게 쓰이는지는 개념으로 정리하는 것으로 마무리합니다.

## 12. 정리 — 실습 코드 1 + 2를 합친 전체 그림

```
                 [실습 코드 1: SL-CAI]                    [실습 코드 2: RL-CAI 앞부분]
사용자 질문 ──▶ 초기 응답 ──▶ Self-Critique ──▶ Revision      여러 응답 후보 ──▶ Pairwise 비교 ──▶ 선호도 데이터
                                                    │                                              │
                                                    └──────────────▶ (파인튜닝 데이터로 사용) ◀─────┘
                                                                            │
                                                                            ▼
                                                         (개념) 보상 모델 학습 → PPO 강화학습
                                                                            │
                                                                            ▼
                                                              더 헌법에 부합하는 최종 모델
```

### ✅ 이번 실습에서 배운 것
- 같은 질문에 대해 여러 응답 후보를 만들고(`generate_candidate_responses`),
  이를 헌법에 비추어 비교(`compare_responses`)한 뒤, `chosen`/`rejected` 형태의
  표준적인 선호도 데이터로 정리(`build_preference_example`)하는 전체 과정을 구현했습니다.
- "비판(critique)"과 "비교(compare)"는 프롬프트 설계상 매우 비슷하지만,
  비교는 결과를 코드로 파싱해야 한다는 점에서 한 단계 더 나아간 작업이었습니다.
- 모델의 자유 형식 출력에서 원하는 정보만 뽑아내기 위해, 프롬프트에서
  출력 형식을 명시적으로 지정하고 그 형식을 기준으로 파싱하는 패턴을 익혔습니다.

### ⚠️ 한계
- `compare_responses()` 의 파싱 로직은 모델이 요청한 형식을 정확히 지킨다는 가정에
  의존합니다. 실무에서는 형식을 벗어난 응답을 더 안전하게 처리하는 로직
  (예: 재시도, 정규표현식 강화, 별도 검증 모델 사용 등)을 추가하는 것이 좋습니다.
- 이번 실습의 비교는 항상 "헌법 전체를 기준으로 한 뭉뚱그려진 판단"입니다.
  실제 연구에서는 원칙별로 세분화해서 비교하거나, 여러 번 비교해 평균을 내는 등
  더 정교한 방법을 사용하기도 합니다.

### ✏️ 직접 해보기 (연습 문제)
1. `generate_candidate_responses()` 의 `n` 을 3~4로 늘리고, 모든 쌍을 서로 비교해서
   가장 많이 "선택된" 응답을 찾는 코드를 작성해보세요. (힌트: `itertools.combinations` 사용)
2. `compare_responses()` 의 파싱이 실패했을 때(`preferred is None`), 기본값을 쓰는 대신
   해당 예시를 데이터셋에서 아예 제외하도록 `build_preference_example()` 을 수정해보세요.
3. 헌법의 5가지 원칙 중 하나만 골라서(예: "정직성"만) 비교하도록 `compare_responses()` 의
   프롬프트를 수정해보고, 결과가 헌법 전체를 기준으로 비교했을 때와 어떻게 달라지는지 비교해보세요.